<a href="https://colab.research.google.com/github/shin-noda/leetcode-problemset-97/blob/main/Problem3454.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [244]:
class SegmentTree:
    def __init__(self, x_coordinates):
        self.x_coords = x_coordinates
        self.num_intervals = len(x_coordinates) - 1
        self.cover_count = [0] * (4 * self.num_intervals)
        self.total_length = [0] * (4 * self.num_intervals)


    def update(self, query_left, query_right, active_delta):
        self._update_node(1, 0, self.num_intervals, query_left, query_right, active_delta)


    def get_current_active_length(self):
        return self.total_length[1]


    def _update_node(self, node, segment_left, segment_right, query_left, query_right, delta):
        if query_right <= segment_left or segment_right <= query_left:
            return

        if query_left <= segment_left and segment_right <= query_right:
            self.cover_count[node] += delta

        else:
            mid = (segment_left + segment_right) // 2
            self._update_node(node * 2, segment_left, mid, query_left, query_right, delta)
            self._update_node(node * 2 + 1, mid, segment_right, query_left, query_right, delta)

        if self.cover_count[node] > 0:
            self.total_length[node] = self.x_coords[segment_right] - self.x_coords[segment_left]

        else:
            if segment_right - segment_left == 1:
                # Base case: it's a leaf interval and nothing covers it
                self.total_length[node] = 0

            else:
                # Internal node: sum up the lengths of both child segments
                self.total_length[node] = self.total_length[node * 2] + self.total_length[node * 2 + 1]


class Solution:
    def separateSquares(self, squares):
        total_area = 0
        half_area = 0
        int_y = 0
        frac = 0.0

        # This stores: y_val: partial_area_at_y
        partial_area = {}

        # Parse geometric boundaries into unique events and tracking points
        sweep_events = []
        x_points = set()
        for x, y, l in squares:
            # Square enters the sweep line
            sweep_events.append((y, x, x + l, 1))

            # Square leaves the sweep line
            sweep_events.append((y + l, x, x + l, -1))

            x_points.add(x)
            x_points.add(x + l)

        # Coordinate compression for x-axis to map massive coordinate values to dense indices
        sorted_x_coords = sorted(x_points)
        x_to_index = {x_val: idx for idx, x_val in enumerate(sorted_x_coords)}

        # Sort the horizontal sweep line events by their y-coordinate moving bottom-to-top
        sweep_events.sort()

        # Initialize the segment treee with our compressed layout
        tree = SegmentTree(sorted_x_coords)

        previous_y = sweep_events[0][0]

        # Initialize 'partial_area'
        partial_area[previous_y] = 0

        for current_y, x1, x2, active_delta in sweep_events:
            # Calculate the height of the horizontal slice we just passed through
            slice_height = current_y - previous_y

            total_area += slice_height * tree.get_current_active_length()

            # Update the current partial area
            partial_area[current_y] = total_area

            tree.update(
                query_left=x_to_index[x1],
                query_right=x_to_index[x2],
                active_delta=active_delta
            )

            previous_y = current_y

        # This is the target area
        half_area = total_area / 2.0

        # Do a 2-stage binary search
        y_coords = list(partial_area.keys())
        left = 0
        right = len(y_coords) - 1
        while left <= right:
            mid = (left + right) // 2
            mid_area = partial_area[y_coords[mid]]

            if mid_area < half_area:
                left = mid + 1

            # Note: this includes the case
            # abs(mid_area - half_area) <= threshold
            # but we might do better
            else:
                right = mid - 1

        # Switch left and right
        left, right = right, left

        # Updte left and right
        left, right = y_coords[left], y_coords[right]

        # Int part of the area
        int_y = left

        # Left, Right, and Half areas
        l_a = partial_area[left]
        r_a = partial_area[right]

        # m and b: y = mx + b
        # frac = x and half_area = y
        # x = (y - b) / m
        m = (r_a - l_a) / (right - left)
        b = l_a

        frac = (half_area - b) / m

        return float(int_y) + frac

In [245]:
solution = Solution()

In [246]:
squares = [[0,0,1],[2,2,1]]

print(solution.separateSquares(squares))

1.0


In [247]:
squares = [[0,0,2],[1,1,1]]

print(solution.separateSquares(squares))

1.0


In [248]:
squares = [[0,0,1]]

print(solution.separateSquares(squares))

0.5


In [249]:
squares = [[0,0,1],[1,1,2]]

# Expected: 1.75
print(solution.separateSquares(squares))

1.75


In [250]:
squares = [[15,21,2],[19,21,3]]

# Expected: 22.3
print(solution.separateSquares(squares))

22.3


In [251]:
squares = [[5,23,1],[6,30,1]]

# Expected: 24.0
print(solution.separateSquares(squares))

24.0


In [252]:
squares = [[0,0,9],[10,0,9],[0,10,1],[0,10,9],[10,10,9]]

# Expected: 9.0
print(solution.separateSquares(squares))

9.0
